# data-cleaning_002

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (104).ipynb`

| Field | Value |
|---|---|
| Section | `data_cleaning` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `daily` |
| Code cells | 1 |
| Detected functions | preprocess_ohlc, create_quarterly_features, add_tradebook_features, step_forward_prediction_with_top_10_tickers |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Load and preprocess OHLC data
def preprocess_ohlc(ohlc_data):
    ohlc_data['Date'] = pd.to_datetime(ohlc_data['Date'])
    ohlc_data.set_index('Date', inplace=True)
    return ohlc_data

# Step 2: Feature engineering from OHLC
def create_quarterly_features(df):
    grouped = df.groupby('Ticker')
    feature_list = []

    for ticker, group in grouped:
        daily = group.resample('1D').agg({
            'open': 'first',
            'high': 'max',
            'low': 'min',
            'close': 'last',
            'volume': 'sum'
        }).dropna()

        daily['return'] = daily['close'].pct_change()
        daily['volatility'] = daily['return'].rolling(21).std()
        daily['rolling_sharpe'] = daily['return'].rolling(21).mean() / (daily['volatility'] + 1e-6)

        monthly = daily.resample('M').agg({
            'return': ['mean', 'std'],
            'rolling_sharpe': 'mean'
        })
        monthly.columns = ['monthly_return_mean', 'monthly_return_std', 'monthly_rolling_sharpe']
        monthly['monthly_pnl_pct'] = (1 + monthly['monthly_return_mean']) ** 21 - 1

        quarterly = daily.resample('Q').agg({
            'close': 'last',
            'return': ['mean', 'std'],
            'rolling_sharpe': 'mean',
            'volume': 'mean'
        })

        quarterly.columns = ['_'.join(col).strip() for col in quarterly.columns]
        quarterly['ticker'] = ticker
        quarterly['forward_return'] = quarterly['close_last'].pct_change().shift(-1)

        # Merge monthly stats into the quarter end
        monthly['quarter'] = monthly.index.to_period('Q')
        quarterly['quarter'] = quarterly.index.to_period('Q')
        monthly_features = monthly.groupby('quarter').mean()
        quarterly = quarterly.join(monthly_features, on='quarter')

        feature_list.append(quarterly)

    return pd.concat(feature_list).dropna().reset_index()

# Step 3: Add tradebook-based features
def add_tradebook_features(features_df, tradebook_df):
    tradebook_df['Entry Time'] = pd.to_datetime(tradebook_df['Entry Time'])
    tradebook_df['quarter'] = tradebook_df['Entry Time'].dt.to_period('Q')

    agg = tradebook_df.groupby(['Ticker', 'quarter']).agg({
        'PnL': ['mean', 'std'],
        'Direction': lambda x: (x == 'long').mean(),
        'Entry Time': 'count'
    }).reset_index()

    agg.columns = ['Ticker', 'quarter', 'pnl_mean', 'pnl_std', 'long_ratio', 'trade_count']
    agg['Date'] = agg['quarter'].dt.end_time
    agg.drop(columns='quarter', inplace=True)

    features_df = pd.merge(features_df, agg, left_on=['ticker', 'Date'], right_on=['Ticker', 'Date'], how='left')
    features_df.drop(columns='Ticker', inplace=True)
    features_df.fillna(0, inplace=True)
    return features_df

# Step 4: Step-forward prediction with top 10 tickers and accuracy metrics (MSE, MAE, R²)
def step_forward_prediction_with_top_10_tickers(features_df):
    features_df.sort_values(by=['ticker', 'Date'], inplace=True)
    unique_quarters = sorted(features_df['Date'].unique())

    results = []  # To store the results for later analysis or plotting
    all_actual = []  # To store actual returns for performance metrics
    all_predicted = []  # To store predicted returns for performance metrics

    # Start from the 4th quarter and move forward
    for i in range(3, len(unique_quarters)):
        # Train on all available quarters up to the current one, and predict the next one
        train_quarters = unique_quarters[:i]  # All data up to the current quarter
        test_quarter = unique_quarters[i]  # The next quarter to predict

        train_df = features_df[features_df['Date'].isin(train_quarters)]
        test_df = features_df[features_df['Date'] == test_quarter]

        # Prepare features and target variable for training
        X_train = train_df.drop(columns=['Date', 'ticker', 'forward_return', 'quarter'])
        y_train = train_df['forward_return']
        X_test = test_df.drop(columns=['Date', 'ticker', 'forward_return', 'quarter'])

        # Apply scaling only to numeric columns
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Initialize and train the XGBoost model
        model = XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
        model.fit(X_train_scaled, y_train)

        # Predict the next quarter's return
        predictions = model.predict(X_test_scaled)

        # Store the results for comparison
        result_df = test_df[['Date', 'ticker']].copy()
        result_df['predicted_return'] = predictions
        result_df['actual_return'] = test_df['forward_return'].values

        # Sort by predicted return and select top 10 tickers
        top_10_df = result_df.sort_values(by='predicted_return', ascending=False).head(10)

        # Print the top 10 results for each quarter
        print(f"Top 10 Tickers for quarter {test_quarter}:")
        print(top_10_df[['Date', 'ticker', 'predicted_return', 'actual_return']])
        print("-" * 50)  # Separator for clarity

        # Append results for later use (if needed)
        results.append(top_10_df)

        # Collect actual and predicted values for accuracy metrics
        all_actual.extend(test_df['forward_return'].values)
        all_predicted.extend(predictions)

    # Convert lists to arrays for metrics calculation
    all_actual = np.array(all_actual)
    all_predicted = np.array(all_predicted)

    # Calculate performance metrics
    mse = mean_squared_error(all_actual, all_predicted)
    mae = mean_absolute_error(all_actual, all_predicted)
    r2 = r2_score(all_actual, all_predicted)

    # Print metrics
    print("\nPerformance Metrics:")
    print(f"Mean Squared Error (MSE): {mse}")
    print(f"Mean Absolute Error (MAE): {mae}")
    print(f"R-squared (R²): {r2}")

    # Return all results for further analysis if necessary
    return pd.concat(results).reset_index(drop=True)

# Example usage (after loading your data):
ohlc_data = pd.read_csv('/content/drive/MyDrive/3year_5min_data.csv')
tradebook_data = pd.read_csv('/content/drive/MyDrive/tradebook_adanient.csv')

# Preprocess the data
ohlc_data = preprocess_ohlc(ohlc_data)

# Create quarterly features
features_df = create_quarterly_features(ohlc_data)

# Add tradebook features
features_df = add_tradebook_features(features_df, tradebook_data)

# Perform step-forward prediction with top 10 tickers and metrics
results = step_forward_prediction_with_top_10_tickers(features_df)

# Check the results
print(results.head())
